In [1]:
import numpy as np
import tensorflow as tf
import pickle

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Load IMDB RAW TEXT


num_words = 10000
max_len = 500

# Load raw dataset (not pre-encoded)
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=num_words)


d:\DL Project\venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [3]:
word_index = imdb.get_word_index()

In [4]:
# Convert integer sequences back to words
reverse_word_index = {v: k for k, v in word_index.items()}

In [5]:
def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in encoded_review])

x_train_text = [decode_review(review) for review in x_train]
x_test_text = [decode_review(review) for review in x_test]


In [ ]:
# Tokenizer 

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train_text)

In [7]:
x_train_seq = tokenizer.texts_to_sequences(x_train_text)
x_test_seq = tokenizer.texts_to_sequences(x_test_text)

x_train_pad = pad_sequences(x_train_seq, maxlen=max_len)
x_test_pad = pad_sequences(x_test_seq, maxlen=max_len)

In [8]:
# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
# Building the  Model 

model = Sequential([
    Embedding(num_words, 128, input_length=max_len),
    LSTM(128),
    Dense(1, activation="sigmoid")
])

d:\DL Project\venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [12]:
history = model.fit(
    x_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 456s 722ms/step - accuracy: 0.7618 - loss: 0.4878 - val_accuracy: 0.7732 - val_loss: 0.4640
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 429s 686ms/step - accuracy: 0.8461 - loss: 0.3582 - val_accuracy: 0.8364 - val_loss: 0.3729
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 364s 582ms/step - accuracy: 0.9151 - loss: 0.2183 - val_accuracy: 0.8704 - val_loss: 0.3212
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 345s 522ms/step - accuracy: 0.9459 - loss: 0.1506 - val_accuracy: 0.8752 - val_loss: 0.3361
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 819s 1s/step - accuracy: 0.9625 - loss: 0.1095 - val_accuracy: 0.8722 - val_loss: 0.4196
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 406s 563ms/step - accuracy: 0.9812 - loss: 0.0606 - val_accuracy: 0.8684 - val_loss: 0.4900


In [14]:
model.save("sentiment_model.h5")

print("Training complete. Model and tokenizer saved.")

Training complete. Model and tokenizer saved.
